# Import

In [33]:
import pandas as pd

# Load all cleaning functions
from gene_cleanup_nb_02 import *

# Files

## Input

In [34]:
# Load the main 'abstract' dataset
abstract_df = pd.read_pickle("Datos_largo_test_3_7_oct_2.pickle")
abstract_df.head()

,0
Increased production of zeaxanthin and other pigments by application of genetic engineering techniques to Synechocystis sp. strain PCC 6803.,[(overexpression of crtR induced a 25-fold inc...
Expression of Alcaligenes eutrophus flavohemoprotein and engineered Vitreoscilla hemoglobin-reductase fusion protein for improved hypoxic growth of Escherichia coli.,None
Environmental biotechnology.,None
Altered regulation of pyruvate kinase or co-overexpression of phosphofructokinase increases glycolytic fluxes in resting Escherichia coli.,[(overexpression of E. coli phosphofructokinas...
A novel genetically engineered pathway for synthesis of poly(hydroxyalkanoic acids) in Escherichia coli.,[(expressing butyrate kinase (buk) phosphotran...


In [35]:
# Load multiple pickle segments and concatenate
combined_parts = []

for idx in range(10):
    part = pd.read_pickle(f"3Datos_largo_7_oct_pt{idx+1}.pickle")
    combined_parts.append(part)

all_papers_df = pd.concat(combined_parts)
all_papers_df.head()

,0
"Formation of functional heterologous complexes using subunits from the picromycin, erythromycin and oleandomycin polyketide synthases.",[(of PKS engineering strategies from domain an...
Engineering desiccation tolerance in Escherichia coli.,[(arrow in lane B indicates the position of su...
"Cloning, nucleotide sequence, and heterologous expression of the biosynthetic gene cluster for R1128, a non-steroidal estrogen receptor antagonist. Insights into an unusual priming mechanism.",[(whereas its carrying capacity as a cosmid cl...
The biosynthetic gene cluster for the antitumor drug bleomycin from Streptomyces verticillus ATCC15003 supporting functional interactions between nonribosomal peptide synthetases and a polyketide synthase.,[(co-introduction of both pbs9 and pbs12 into ...
Cofactor regeneration by a soluble pyridine nucleotide transhydrogenase for biological production of hydromorphone.,[(NADPH and NAD supplied at a of 02 mM. 1 the...


## Output

In [37]:
# Output file for final modifications list
output_file = "Cleaned_modifications.json"

# Cleanup

In [ ]:
# Drop NA rows, then remove known metabolites
abstract_clean = abstract_df.dropna().apply(remove_known_metabolites)
all_papers_clean = all_papers_df.dropna().apply(remove_known_metabolites)

abstract_clean.head()

In [ ]:
# Remove uppercase pathway-level entities (not real gene symbols)
abstract_clean = abstract_clean.apply(clean_uppercase_pathway_genes)
all_papers_clean = all_papers_clean.apply(clean_uppercase_pathway_genes)

# Remove protein names / non-gene products
abstract_clean = abstract_clean.apply(clean_non_gene_products)
all_papers_clean = all_papers_clean.apply(clean_non_gene_products)


In [40]:
abstract_clean.columns = ["Modifications"]
all_papers_clean.columns = ["Modifications"]

In [41]:
# Basic cleanup on individual modifications (before negation fix)
abstract_clean["Modifications"] = abstract_clean["Modifications"].apply(clean_modifications_list)
all_papers_clean["Modifications"] = all_papers_clean["Modifications"].apply(clean_modifications_list)

# Remove rows that did not yield valid modification lists
abstract_clean = abstract_clean.dropna()
all_papers_clean = all_papers_clean.dropna()

In [ ]:
abstract_clean["Modifications_Corrected"] = abstract_clean["Modifications"].apply(
    fix_negation_mismatches_in_modifications
)

all_papers_clean["Modifications_Corrected"] = all_papers_clean["Modifications"].apply(
    fix_negation_mismatches_in_modifications
)

# Analysis

In [43]:
# Extract modification sets from each paper's corrected modification list
abstract_mods = extract_paper_modifications(abstract_clean, "Modifications_Corrected")
paper_mods = extract_paper_modifications(all_papers_clean, "Modifications_Corrected")

# Combine both
all_modifications = abstract_mods | paper_mods

len(all_modifications)

8352

In [44]:
# Convert modification dictionary into DataFrame
mods_df = pd.DataFrame([all_modifications]).T
mods_df.columns = ["Modifications"]

# Replace empty dicts with None
mods_df["Modifications"] = mods_df["Modifications"].apply(lambda x: None if x == {} else x)

mods_df.head()

,Modifications
Increased production of zeaxanthin and other pigments by application of genetic engineering techniques to Synechocystis sp. strain PCC 6803.,"{ipi -> expression, crtP -> was introduced, cr..."
Altered regulation of pyruvate kinase or co-overexpression of phosphofructokinase increases glycolytic fluxes in resting Escherichia coli.,{pyk -> overexpression}
A novel genetically engineered pathway for synthesis of poly(hydroxyalkanoic acids) in Escherichia coli.,"{phaC -> expressing, buk -> expressing, ptb ->..."
Properties of engineered poly-3-hydroxyalkanoates produced in recombinant Escherichia coli strains.,"{PhbB -> introducing, phbB -> introduction}"
Metabolic engineering of carotenoid accumulation in Escherichia coli by modulation of the isoprenoid precursor pool with expression of deoxyxylulose phosphate synthase.,"{dxs -> overexpression, DXS -> not overexpress..."


In [48]:

mods_df.explode("Modifications").Modifications.str.split("->").str[0].str.strip().value_counts().head(20)

Modifications
pta      265
ldh      262
ldhA     228
zwf      192
ppc      184
pgi      168
PKS      166
adhE     165
LDH      165
PTS      163
ERG20    146
PPP      142
ptsG     139
acs      138
cre      135
lacZ     133
ackA     127
ERG9     126
dxs      124
gltA     123
Name: count, dtype: int64

# Final cleanup

In [ ]:
rows_with_problems = []

for title, mod_set in mods_df["Modifications"].items():
    if mod_set is None:
        continue
    
    original = mod_set.copy()
    cleaned_set, changed, change_log = clean_mod_set(mod_set)

    if changed:
        rows_with_problems.append(title)
        
        print(f"Title: {title}")
        print("BEFORE:", original)
        print("AFTER :", cleaned_set)
        print("Changes:")
        for change in change_log:
            print(" -", change)
        print("-" * 50)

    mods_df.at[title, "Modifications"] = cleaned_set

print("\nRows with issues:")
for row in rows_with_problems:
    print(" -", row)

In [47]:
mods_df.to_json(output_file)